*_Week2 assignment_*

Part 1 Exercises (MLP)

    E01: Tune the hyperparameters of the training to beat Andrej's best validation loss of 2.2.
    E02: Initialization matters.
        What is the loss you'd get if the predicted probabilities at initialization were perfectly uniform? What loss do we actually achieve?
        Can you tune the initialization to get a starting loss that is much more similar to the uniform loss?
    E03: Read the Bengio et al. 2003 paper (link above). Implement and try any idea from the paper. Did it work?

Part 2 Exercises (Internals & BatchNorm)

    E01: I did not get around to seeing what happens when you initialize all weights and biases to zero. Try this and train the neural net.
        You might think either that 1) the network trains just fine or 2) the network doesn't train at all, but actually it is 3) the network trains but only partially, and achieves a pretty bad final performance.
        Task: Inspect the gradients and activations to figure out what is happening and why the network is only partially training, and what part is being trained exactly.
    E02: BatchNorm has the big advantage that after training, the parameters (gamma/beta) can be "folded into" the weights of the preceding Linear layers, erasing the need to forward it at test time.
        Task: Set up a small 3-layer MLP with batchnorms, train the network, then "fold" the batchnorm gamma/beta into the preceding Linear layer's W,b by creating a new W2, b2 and erasing the batch norm. Verify that this gives the same forward pass during inference.

In [13]:
#-------SETUP---------

In [3]:
#fahh lets make it
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [5]:
#opening the names.txt and creating words that contain it
words = open('names.txt', 'r').read().splitlines()

In [6]:
#pre-work creation of the required variables
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)

In [7]:
# build the dataset
block_size = 4 # context length: how many characters do we take to predict the next one?
# block_size from 3 to 4

def build_dataset(words):  
  X, Y = [], []
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] 

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 4]) torch.Size([182625])
torch.Size([22655, 4]) torch.Size([22655])
torch.Size([22866, 4]) torch.Size([22866])


P1 E02 If predicted probabilities are same then the probability of the outcomes will be (1/vocab_size)

In [16]:
# P1 E02
import math
uniformprob_loss = -math.log(1/vocab_size)
print(uniformprob_loss)

3.295836866004329


In [14]:
#-------------INITIALIZATION------------

In [36]:
n_embd = 15
n_hidden = 300
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g)
b1 = torch.randn(n_hidden, generator=g) 
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01  
b2 = torch.randn(vocab_size, generator=g) * 0
#P1 E02 W2 scaled down by 0.01 to get intial loss comaprable to the equal prob case loss
parameters = [C, W1, b1, W2, b2]

In [33]:
# P1 E02
# initial loss over a 50 size batch 
emb = C[Xtr[:50]]
h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)
logits = h @ W2 + b2
initial_loss = F.cross_entropy(logits, Ytr[:50])
initial_loss.item()

30.9421329498291

In [28]:
#--------TRAINING----------

In [37]:
for p in parameters:
    p.requires_grad = True

In [38]:
max_steps = 200000
batch_size = 64 # now training with large batches

for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])

    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i % 20000 == 0:
        print(f"{i:7d}/{max_steps}: {loss.item():.4f}")

      0/200000: 3.2835
  20000/200000: 2.2033
  40000/200000: 2.4428
  60000/200000: 2.0163
  80000/200000: 2.0354
 100000/200000: 1.9788
 120000/200000: 2.1339
 140000/200000: 1.9061
 160000/200000: 2.0233
 180000/200000: 2.0871


Achieved the loss of 3.2835 in first iteration which is similar to equi prob loss (3.295836866004329)

In [39]:
#LOSS calculation
@torch.no_grad()
def split_loss(split):
    x, y = {'train': (Xtr, Ytr), 'val': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    emb = C[x]
    h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    print(f"{split}: {loss.item():.4f}")

split_loss('train')
split_loss('val')

train: 1.9539
val: 2.0924


Changing the hyperparams to n_embd = 15, n_hidden = 300 and block_size = 4 we get a final loss of 2.0924 (beated 2.2 loss)

In [33]:
#P1 E03 implementations
n_embd = 15
n_hidden = 300
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd*block_size)**0.5)
b1 = torch.randn(n_hidden, generator=g) 
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01  
b2 = torch.randn(vocab_size, generator=g) * 0
W_direct = torch.randn(n_embd, vocab_size, generator=g) * 0.01
#P1 E02 W2 scaled down by 0.01 to get intial loss comaprable to the equal prob case loss
parameters = [C, W1, b1, W2, b2, W_direct]

In [34]:
for p in parameters:
    p.requires_grad = True

In [35]:
C[Xtr[:32]].shape

torch.Size([32, 4, 15])

In [36]:
max_steps = 200000
batch_size = 64 # now training with large batches

for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)
    logits = (h @ W2 + b2) + emb.mean(dim=1) @ W_direct 
    fahh = 0.0001
    loss = F.cross_entropy(logits, Ytr[ix]) + fahh * sum(p.pow(2).mean() for p in [W1, W2]) 
    # L2 regularization
    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i % 20000 == 0:
        print(f"{i:7d}/{max_steps}: {loss.item():.4f}")

      0/200000: 3.2841
  20000/200000: 2.0426
  40000/200000: 2.1267
  60000/200000: 1.7751
  80000/200000: 1.7985
 100000/200000: 1.5574
 120000/200000: 2.2126
 140000/200000: 1.7573
 160000/200000: 1.7853
 180000/200000: 1.9878


In [29]:
#LOSS calculation
@torch.no_grad()
def split_loss(split):
    x, y = {'train': (Xtr, Ytr), 'val': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    emb = C[x]
    h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    print(f"{split}: {loss.item():.4f}")

split_loss('train')
split_loss('val')

train: 2.4992
val: 2.6938


I guess it does not helped tho 


In [39]:
#P2 E01
n_embd = 15
n_hidden = 300
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.zeros((n_embd * block_size, n_hidden))
b1 = torch.zeros((n_hidden)) 
W2 = torch.zeros((n_hidden, vocab_size)) * 0.01  
b2 = torch.zeros((vocab_size)) * 0
#P1 E02 W2 scaled down by 0.01 to get intial loss comaprable to the equal prob case loss
parameters = [C, W1, b1, W2, b2]

In [40]:
for p in parameters:
    p.requires_grad = True

In [43]:
max_steps = 200000
batch_size = 64 # now training with large batches

for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])

    for p in parameters:
        p.grad = None
    loss.backward()
    
    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i % 20000 == 0:
        print(f"{i:7d}/{max_steps}: {loss.item():.4f}")

    print("h mean value:", h.mean().item())      
    print("h max value:", h.max().item())        
    print("h min value:", h.min().item())
    print("W1.grad max abs value:", W1.grad.abs().max().item()) 
    print("b1.grad max abs value:", b1.grad.abs().max().item()) 
    print("W2.grad max abs value:", W2.grad.abs().max().item()) 
    print("b2.grad max abs value:", b2.grad.abs().max().item()) 
    break

      0/200000: 2.6407
h mean value: 0.0
h max value: 0.0
h min value: 0.0
W1.grad max abs value: 0.0
b1.grad max abs value: 0.0
W2.grad max abs value: 0.0
b2.grad max abs value: 0.05697544291615486


In [42]:
#LOSS calculation
@torch.no_grad()
def split_loss(split):
    x, y = {'train': (Xtr, Ytr), 'val': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    emb = C[x]
    h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    print(f"{split}: {loss.item():.4f}")

split_loss('train')
split_loss('val')

train: 2.8227
val: 2.8210


Trained yet tho //
loss is not zero bcuz of the softmax function it gives almost equal probs to each and 3.3 is intial loss //
only b2 has got the grads and the remaining weights and biases are dead due to zero grad and all so they are not trained so finally only b2 (bias) is only trained

In [44]:
#P2 E02 

In [63]:
import torch
import torch.nn.functional as F

class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None
    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None: 
            self.out += self.bias
        return self.out
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)
    def __call__(self, x):
        if self.training:
            xmean = x.mean(0, keepdim=True)
            xvar = x.var(0, keepdim=True)
        else:
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        return self.out
    def parameters(self):
        return [self.gamma, self.beta]

class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    def parameters(self):
        return []

n_embd = 10 
n_hidden = 100 
g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, n_embd), generator=g)

layers = [
  Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(n_hidden, vocab_size, bias=False)
]

parameters = [p for layer in layers for p in layer.parameters()]
parameters.append(C)

for p in parameters:
    p.requires_grad = True

lossi = []
ud = []

In [69]:
max_steps = 200000
batch_size = 32

# Ensure network starts in training mode so running stats update actively
for layer in layers:
    if hasattr(layer, 'training'):
        layer.training = True

for i in range(max_steps):
  
  # Minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix]
  
  # Forward pass
  emb = C[Xb] 
  x = emb.view(emb.shape[0], -1) 
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, Yb)
  
  # Backward pass 
  if i == 0:
    for layer in layers:
      if hasattr(layer, 'out') and layer.out.requires_grad:
        layer.out.retain_grad()

  for p in parameters:
    p.grad = None
  loss.backward()
  
  # Update step with learning rate decay
  lr = 0.1 if i < 150000 else 0.01 
  for p in parameters:
    p.data += -lr * p.grad

  # Track performance statistics
  if i % 10000 == 0: 
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())
  
  with torch.no_grad():
    ud.append([((lr*p.grad).std() / p.data.std()).log10().item() if p.grad is not None else 0 for p in parameters])

  if i >= 1000:
    break

      0/ 200000: 2.4562


In [70]:
# 1. Flip BatchNorm layers to eval mode so they use frozen running statistics
for layer in layers:
    if hasattr(layer, 'training'):
        layer.training = False

Xb_test = Xtr[:10]
Yb_test = Ytr[:10]

emb_test = C[Xb_test]
X_test = emb_test.view(emb_test.shape[0], -1)

x = X_test
for layer in layers:
    x = layer(x)
original_logits = x
original_loss = F.cross_entropy(original_logits, Yb_test)

folded_layers = []
i = 0
while i < len(layers):

    if i + 1 < len(layers) and layers[i].__class__.__name__ == 'Linear' and layers[i+1].__class__.__name__ == 'BatchNorm1d':
        linear_layer = layers[i]
        bn_layer = layers[i+1]
        
        W = linear_layer.weight
        b = linear_layer.bias if linear_layer.bias is not None else torch.zeros(W.shape[1])
        
        gamma = bn_layer.gamma
        beta = bn_layer.beta
        rmean = bn_layer.running_mean
        rvar = bn_layer.running_var
        eps = bn_layer.eps
        
        sigma_all = torch.sqrt(rvar + eps)
        scale = gamma / sigma_all
        
        W_fold = W * scale
        b_fold = ((b - rmean) / sigma_all) * gamma + beta
        
        fused_linear = Linear(W.shape[0], W.shape[1], bias=True)
        fused_linear.weight = W_fold
        fused_linear.bias = b_fold
        
        folded_layers.append(fused_linear)
        i += 2 
    else:
        folded_layers.append(layers[i])
        i += 1

x_fold = X_test
for layer in folded_layers:
    x_fold = layer(x_fold)
folded_logits = x_fold
folded_loss = F.cross_entropy(folded_logits, Yb_test)

print(f"Original network loss:       {original_loss.item():.6f}")
print(f"Folded network loss:         {folded_loss.item():.6f}")

Original network loss:       2.733044
Folded network loss:         2.733044
